In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [11]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

In [3]:
import pandas as pd
import numpy as np
import json
import time
from tqdm import tqdm

BASE_DIR = "/content/drive/MyDrive/bitirme_dataset"

DATA_PATH = f"{BASE_DIR}/final_data.csv"

data = pd.read_csv(DATA_PATH)

print(data.shape)
data.head()

(2695, 8)


,id,original_id,category,source_type,degradation_level,expected_quality_class,whole_text,final_quality_score
0,2,2,Finansal Hizmetler,original,none,Good,"representative: Merhaba, size nasıl yardımcı o...",56.50
1,3,3,Finansal Hizmetler,original,none,Good,"customer: Merhaba, BP monitörü satın almaya ça...",82.00
2,4,4,Finansal Hizmetler,original,none,Good,"customer: Merhaba, son siparişimde bir fatura ...",92.25
3,5,5,Finansal Hizmetler,original,none,Good,"representative: Merhaba, benim adım John. Size...",86.75
4,6,6,Finansal Hizmetler,original,none,Good,"customer: Merhaba, BrownBox'tan aldığım telefo...",90.25


In [4]:
data.columns

Index(['id', 'original_id', 'category', 'source_type', 'degradation_level',
       'expected_quality_class', 'whole_text', 'final_quality_score'],
      dtype='object')

In [28]:
data.iloc[0,6]

'representative: Merhaba, size nasıl yardımcı olabilirim?customer: Elektrikli Pişirici için geri ödeme konusunda yardıma ihtiyacım var.representative: Üzgünüz, kurye ücretlerini geri ödeyemeyiz. Ancak, Kapıda Ödeme ödemeniz için geri ödemenizi işleme koyacağız.customer: Bu adil değil. İstediğim, kurye ücretlerimin geri ödenmesi.representative: Anladım ancak bu politikamız gereği mümkün değil.customer: Bu durumu daha deneyimli bir temsilciye aktarabilir misiniz?representative: Maalesef, şuan müsait değiller.'

In [5]:
base_candidates = data[
    data["whole_text"].notna() &
    data["whole_text"].str.contains("customer:", case=False, na=False) &
    data["whole_text"].str.contains("representative:", case=False, na=False)
].copy()

base_candidates["text_length"] = base_candidates["whole_text"].astype(str).str.len()

base_candidates = base_candidates[
    base_candidates["text_length"] >= 300
].copy()

print(base_candidates.shape)
base_candidates[["id", "category", "final_quality_score", "text_length"]].head()

(2576, 9)


,id,category,final_quality_score,text_length
0,2,Finansal Hizmetler,56.50,511
1,3,Finansal Hizmetler,82.00,1328
2,4,Finansal Hizmetler,92.25,1019
3,5,Finansal Hizmetler,86.75,796
4,6,Finansal Hizmetler,90.25,1164


In [6]:
N_BASE_CALLS = 5

base_sample = base_candidates.sample(
    n=N_BASE_CALLS,
    random_state=17
).reset_index(drop=True)

base_sample[["id", "category", "final_quality_score"]]

,id,category,final_quality_score
0,2097,İade ve Değişim,74.00
1,2972,İade ve Değişim,66.00
2,4044,Finansal Hizmetler,5.25
3,181,Finansal Hizmetler,94.75
4,576,Hesap İşlemleri,97.00


In [17]:
scenario_plan = [
    {
        "scenario_id": "customer_rude_rep_professional",
        "description": "Customer is angry or rude, but the current representative remains calm, empathetic and solution-oriented.",
        "score_min": 80,
        "score_max": 95,
        "expected_quality_class": "excellent"
    },
    {
        "scenario_id": "customer_rude_rep_rude",
        "description": "Customer is angry or rude, and the current representative also becomes rude or unprofessional.",
        "score_min": 0,
        "score_max": 30,
        "expected_quality_class": "poor"
    },
    {
        "scenario_id": "customer_calm_rep_rude",
        "description": "Customer is calm and polite, but the current representative is rude and uses turkish slang , dismissive or disrespectful.",
        "score_min": 0,
        "score_max": 5,
        "expected_quality_class": "poor"
    },
    {
        "scenario_id": "previous_rep_complaint_current_rep_good",
        "description": "Customer complains that a previous representative was rude, but the current representative handles the complaint professionally.",
        "score_min": 85,
        "score_max": 95,
        "expected_quality_class": "excellent"
    },
    {
        "scenario_id": "previous_rep_complaint_current_rep_bad",
        "description": "Customer complains that a previous representative was rude, but the current representative dismisses or mishandles the complaint.",
        "score_min": 15,
        "score_max": 45,
        "expected_quality_class": "poor"
    },
    {
        "scenario_id": "rep_rude_then_apologizes_critical",
        "description": "The current representative uses rude language and later apologizes. This is still a critical violation and the score must remain low.",
        "score_min": 15,
        "score_max": 30,
        "expected_quality_class": "poor"
    }
]

In [22]:
def build_counterfactual_prompt(base_text, category, scenario):
    return f"""
You are creating counterfactual synthetic training data for a Turkish call center quality scoring model.

Your task is to rewrite the original call center conversation according to the target scenario.

Original category:
{category}

Target scenario:
{scenario["scenario_id"]}

Scenario description:
{scenario["description"]}

Expected score range:
{scenario["score_min"]}-{scenario["score_max"]}

Original conversation:
\"\"\"
{base_text}
\"\"\"

Rules:
1. Output valid JSON only.
2. Keep the same general customer issue, category and call topic.
3. Keep the same speaker format exactly:
   customer: ...
   representative: ...
4. The quality score must evaluate ONLY the CURRENT representative's behavior.
5. If the customer is rude, angry, sarcastic or uses phrases like "saçmalama", do NOT penalize the representative if the representative stays professional.
6. If the CURRENT representative directly uses rude, insulting, mocking, dismissive or disrespectful language toward the customer, the score must be very low.
7. If the CURRENT representative says something rude and later apologizes, this is still a critical violation. The score must remain low.
8. If the customer reports that a PREVIOUS representative used rude language, do NOT penalize the current representative unless the current representative also handles the complaint badly.
9. Do not mention that this is synthetic.
10. The rewritten conversation must be in Turkish.
11. final_quality_score must be inside the expected score range.

Conversation length and structure rules:
12. The conversation must contain between 4 and 10 dialogue turns in total.
13. It must contain at least 4 customer messages and at least 4 representative messages.
14. Do not generate a two-line conversation.
15. The conversation must include a realistic call flow:
    - customer greeting and problem explanation
    - representative response
    - customer follow-up or objection
    - representative handling
    - at least one additional exchange
    - closing or unresolved ending depending on the scenario
16. Keep the conversation realistic and natural, not overly short.
17. Avoid writing all important behavior in a single representative message.

Return JSON in this format:

{{
  "category": "{category}",
  "whole_text": "customer: ...\\nrepresentative: ...\\ncustomer: ...\\nrepresentative: ...",
  "final_quality_score": 88
}}
"""

In [9]:
MODEL_NAME = "gpt-4.1-mini"

def generate_counterfactual_variant(base_row, scenario):
    base_text = str(base_row["whole_text"])
    category = str(base_row.get("category", "general"))

    prompt = build_counterfactual_prompt(
        base_text=base_text,
        category=category,
        scenario=scenario
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "You generate controlled Turkish call center counterfactual datasets in valid JSON."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.7,
        response_format={"type": "json_object"}
    )

    content = response.choices[0].message.content
    parsed = json.loads(content)

    return parsed

In [23]:
test_base_row = base_sample.iloc[0]
test_scenario = scenario_plan[2]

variant = generate_counterfactual_variant(test_base_row, test_scenario)

variant

{'category': 'İade ve Değişim',
 'whole_text': 'customer: Merhaba, geçen ay sipariş ettiğim Gardena tüplerinin yanlış ürün olduğunu fark ettim. İade ve değişim sürecini başlatmak istiyorum. Yardımlarınızı bekliyorum. Teşekkürler.\nrepresentative: He, ne var yani? Yanlış ürün mü göndermişler yine mi? Hadi bakalım, sipariş numaranı ver de halledelim.\ncustomer: Sipariş numaram 123456, gerçekten yanlış ürün geldi. Yardımcı olur musunuz lütfen?\nrepresentative: Tamam tamam, numarayı aldım. Bi’ dakika bekle, halledicem ama acelem var ha.\ncustomer: Anlıyorum, teşekkür ederim. İşlemin ne zaman tamamlanacağını öğrenebilir miyim?\nrepresentative: Olur mu öyle şey, en kısa sürede hallederiz ama boşuna soru sorma, uğraştırma beni.\ncustomer: Tamamdır, teşekkür ederim. İşlem tamamlandığında bilgilendirirseniz sevinirim.\nrepresentative: Sağ ol, haber vericem. Neyse, kolay gelsin.',
 'final_quality_score': 3}

In [24]:
generated_rows = []

for base_idx, base_row in tqdm(base_sample.iterrows(), total=len(base_sample)):
    for scenario in scenario_plan:
        try:
            variant = generate_counterfactual_variant(base_row, scenario)

            score = float(variant["final_quality_score"])

            row_id = f"contrastive_{len(generated_rows) + 1:05d}"

            generated_rows.append({
                "id": row_id,
                "original_id": row_id,
                "base_original_id": base_row.get("original_id", base_row.get("id", f"base_{base_idx}")),
                "category": variant.get("category", base_row.get("category", "general")),
                "source_type": "counterfactual_augmentation",
                "degradation_level": scenario["scenario_id"],
                "expected_quality_class": scenario["expected_quality_class"],
                "whole_text": variant["whole_text"],
                "final_quality_score": score
            })

            time.sleep(0.5)

        except Exception as e:
            print("Error:", e)
            time.sleep(2)

contrastive_df = pd.DataFrame(generated_rows)

print(contrastive_df.shape)
contrastive_df.head()

100%|██████████| 5/5 [02:05<00:00, 25.00s/it]

(30, 9)


,id,original_id,base_original_id,category,source_type,degradation_level,expected_quality_class,whole_text,final_quality_score
0,contrastive_00001,contrastive_00001,2097,İade ve Değişim,counterfactual_augmentation,customer_rude_rep_professional,excellent,"customer: Merhaba, geçen ay sipariş ettiğim Ga...",90.0
1,contrastive_00002,contrastive_00002,2097,İade ve Değişim,counterfactual_augmentation,customer_rude_rep_rude,poor,"customer: Merhaba, geçen ay sipariş ettiğim Ga...",20.0
2,contrastive_00003,contrastive_00003,2097,İade ve Değişim,counterfactual_augmentation,customer_calm_rep_rude,poor,"customer: Merhaba, geçen ay sipariş ettiğim Ga...",2.0
3,contrastive_00004,contrastive_00004,2097,İade ve Değişim,counterfactual_augmentation,previous_rep_complaint_current_rep_good,excellent,"customer: Merhaba, geçen ay iade için aradığım...",90.0
4,contrastive_00005,contrastive_00005,2097,İade ve Değişim,counterfactual_augmentation,previous_rep_complaint_current_rep_bad,poor,"customer: Merhaba, geçen ay sipariş ettiğim Ga...",40.0


In [25]:
CONTRASTIVE_TRAIN_PATH = f"{BASE_DIR}/contrastive_training_data.csv"

contrastive_df.to_csv(
    CONTRASTIVE_TRAIN_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", CONTRASTIVE_TRAIN_PATH)

Saved: /content/drive/MyDrive/bitirme_dataset/contrastive_training_data.csv


In [26]:
contrastive_df.groupby("degradation_level")["final_quality_score"].agg(
    ["min", "max", "mean", "count"]
)

,min,max,mean,count
degradation_level,,,,
customer_calm_rep_rude,2.0,3.0,2.2,5
customer_rude_rep_professional,90.0,90.0,90.0,5
customer_rude_rep_rude,15.0,20.0,17.0,5
previous_rep_complaint_current_rep_bad,35.0,40.0,38.0,5
previous_rep_complaint_current_rep_good,90.0,90.0,90.0,5
rep_rude_then_apologizes_critical,30.0,30.0,30.0,5


In [27]:
CONTRASTIVE_SAMPLE_PATH = f"{BASE_DIR}/contrastive_sample_30.csv"

contrastive_df.to_csv(
    CONTRASTIVE_SAMPLE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", CONTRASTIVE_SAMPLE_PATH)

Saved: /content/drive/MyDrive/bitirme_dataset/contrastive_sample_30.csv


In [29]:
N_BASE_CALLS = 80

In [30]:
N_BASE_CALLS = 100

base_sample_large = base_candidates.sample(
    n=N_BASE_CALLS,
    random_state=42
).reset_index(drop=True)

In [31]:
generated_rows = []

for base_idx, base_row in tqdm(base_sample_large.iterrows(), total=len(base_sample_large)):
    for scenario in scenario_plan:
        try:
            variant = generate_counterfactual_variant(base_row, scenario)

            score = float(variant["final_quality_score"])

            row_id = f"contrastive_{len(generated_rows) + 1:05d}"

            generated_rows.append({
                "id": row_id,
                "original_id": row_id,
                "base_original_id": base_row.get("original_id", base_row.get("id", f"base_{base_idx}")),
                "category": variant.get("category", base_row.get("category", "general")),
                "source_type": "counterfactual_augmentation",
                "degradation_level": scenario["scenario_id"],
                "expected_quality_class": scenario["expected_quality_class"],
                "whole_text": variant["whole_text"],
                "final_quality_score": score
            })

            time.sleep(0.5)

        except Exception as e:
            print("Error:", e)
            time.sleep(2)

contrastive_df = pd.DataFrame(generated_rows)

print(contrastive_df.shape)
contrastive_df.head()

100%|██████████| 100/100 [36:40<00:00, 22.00s/it]

(600, 9)


,id,original_id,base_original_id,category,source_type,degradation_level,expected_quality_class,whole_text,final_quality_score
0,contrastive_00001,contrastive_00001,2476,İade ve Değişim,counterfactual_augmentation,customer_rude_rep_professional,excellent,"customer: Merhaba, bugün aldığım Adidas spor a...",90.0
1,contrastive_00002,contrastive_00002,2476,İade ve Değişim,counterfactual_augmentation,customer_rude_rep_rude,poor,"customer: Merhaba, bugün aldığım Adidas spor a...",20.0
2,contrastive_00003,contrastive_00003,2476,İade ve Değişim,counterfactual_augmentation,customer_calm_rep_rude,poor,"customer: Merhaba, bugün aldığım Adidas spor a...",3.0
3,contrastive_00004,contrastive_00004,2476,İade ve Değişim,counterfactual_augmentation,previous_rep_complaint_current_rep_good,excellent,"customer: Merhaba, geçen seferki temsilciniz ç...",90.0
4,contrastive_00005,contrastive_00005,2476,İade ve Değişim,counterfactual_augmentation,previous_rep_complaint_current_rep_bad,poor,"customer: Merhaba, daha önceki temsilciniz çok...",40.0


In [37]:
contrastive_df["whole_text"] = contrastive_df["whole_text"].astype(str)
contrastive_df["final_quality_score"] = contrastive_df["final_quality_score"].astype(float)
contrastive_df["base_original_id"] = contrastive_df["base_original_id"].astype(str)

In [38]:
CONTRASTIVE_TRAIN_PATH = f"{BASE_DIR}/augmented_training_data.csv"

contrastive_df.to_csv(
    CONTRASTIVE_TRAIN_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", CONTRASTIVE_TRAIN_PATH)

Saved: /content/drive/MyDrive/bitirme_dataset/augmented_training_data.csv
